# Donnell-type and Sanders-type kinematics

The library has two elements with the same interpolation and degrees of
freedom, one per kinematics of the cylindrical shell: the Donnell-type
element, namespace `bfsccylinder`, and the Sanders-type element, namespace
`bfsccylinder.sanders`, see
[Kinematics of the cylindrical shell](../kinematics.rst). Both namespaces
give the same names: the element class, the functions `update_*` and the
sizes `*_SPARSE_SIZE` of the sparse arrays. Taking every function of an
analysis from one namespace keeps its matrices consistent with one strain
measure.

This example computes the linear buckling load of the laminated cylinder
Z33 under axial compression with both elements, by the same code, and
compares them with each other and with the buckling load published by
Geier et al. [Geier2002]. The same two analyses are checked by the tests
`tests/test_linear_buckling_Z33_constant_stress.py` and
`tests/test_sanders_linear_buckling_Z33_constant_stress.py`.

**Theory.** The Sanders-type curvatures

$$
\kappa_{yy}^S = -w_{,yy} + \frac{1}{R} v_{,y}, \qquad
\kappa_{xy}^S = -2 w_{,xy} + \frac{3}{2R} v_{,x} - \frac{1}{2R} u_{,y}
$$

add terms in $v$ and $u$ to the Donnell-type ones, $-w_{,yy}$ and
$-2 w_{,xy}$ [Donnell1933], [Sanders1959], [Koiter1960]. The neglected
terms are of relative order $1/n^2$, with $n$ the number of circumferential
waves of the mode. Under axial compression the buckling modes of a
cylinder have many waves, so the two kinematics should give close loads.

In [1]:
import numpy as np
from numpy import isclose, pi
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from numpy.linalg import norm
from scipy.sparse.linalg import eigsh, splu
from composites import laminated_plate

import bfsccylinder
import bfsccylinder.sanders
from bfsccylinder import DOF
from bfsccylinder.quadrature import get_points_weights
from bfsccylinder.utils import assign_constant_ABD

# name of the kinematics: (namespace, element class)
KINEMATICS = {
    'Donnell': (bfsccylinder, bfsccylinder.BFSCCylinder),
    'Sanders': (bfsccylinder.sanders, bfsccylinder.sanders.BFSCCylinderSanders),
}

from IPython.display import Markdown


def table(rows, fmt='{:.5g}'):
    """Markdown table of a list of dicts with the same keys"""
    keys = list(rows[0])
    cell = lambda v: fmt.format(v) if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(keys) + ' |', '|' + '---|'*len(keys)]
    lines += ['| ' + ' | '.join(cell(r[k]) for k in keys) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

## Cylinder Z33

The cylinder Z33 of Castro et al. [Castro2014], Table 4: radius
$R = 250$ mm, length $L = 510$ mm, ten plies of 0.125 mm, the stacking
sequence listed from the inner to the outer surface. The ply material is
Geier1997 of [Castro2014], Table 3, the material of the test cylinders of
Geier and Singh [GeierSingh1997] and of Z33 in [Geier2002]:
$E_{11} = 123.55$ GPa, $E_{22} = 8.708$ GPa, $\nu_{12} = 0.319$,
$G_{12} = 5.695$ GPa. The element uses only the laminate stiffnesses
$\boldsymbol{A}$, $\boldsymbol{B}$, $\boldsymbol{D}$ of the classical
laminated plate theory, see [Constitutive relations](../constitutive.rst);
`laminated_plate` takes six elastic constants, and the two transverse shear
moduli, not used, are given the value of $G_{12}$.

In [2]:
L = 0.510  # m
R = 0.250  # m
b = 2*pi*R  # m, developed circumference

E11 = 123.55e9  # Pa
E22 = 8.708e9
nu12 = 0.319
G12 = 5.695e9
plyt = 0.125e-3  # m
stack = [0, 0, 19, -19, 37, -37, 45, -45, 51, -51]
prop = laminated_plate(stack=stack, plyt=plyt,
                       laminaprop=(E11, E22, nu12, G12, G12, G12))
print('A (N/m) =\n', prop.A.round(0))
print('D (N m) =\n', prop.D.round(2))

A (N/m) =
 [[88730823. 24856986.        0.]
 [24856986. 35066809.        0.]
 [       0.        0. 28478336.]]
D (N m) =
 [[12.31  2.7  -0.35]
 [ 2.7   4.88 -0.71]
 [-0.35 -0.71  3.17]]


## Mesh

The $C^1$ continuity of the element needs a structured mesh of rectangles,
closed around the circumference by giving the last column of nodes the
numbers of the first one, see [Assembling a model](../assembling.rst). 50
nodes around the circumference, the elements nearly square.

In [3]:
ny = 50  # nodes around the circumference
nx = int(ny*L/b)  # nodes along the axis

# node numbers on a grid of nx by ny + 1, the last column closing the cylinder
nids = 1 + np.arange(nx*(ny+1))
nids_mesh = nids.reshape(nx, ny+1)
nids_mesh[:, -1] = nids_mesh[:, 0]
nids = np.unique(nids_mesh)
nid_pos = dict(zip(nids, np.arange(len(nids))))

# coordinates of the nodes on the developed mid-surface
xlin = np.linspace(0, L, nx)
ylin = np.linspace(0, b, ny+1)[:-1]
xmesh, ymesh = np.meshgrid(xlin, ylin, indexing='ij')
x = xmesh.flatten()
y = ymesh.flatten()

# the four nodes of each element, counterclockwise from (x_i, y_j)
n1s = nids_mesh[:-1, :-1].flatten()
n2s = nids_mesh[1:, :-1].flatten()
n3s = nids_mesh[1:, 1:].flatten()
n4s = nids_mesh[:-1, 1:].flatten()
num_elements = len(n1s)
N = DOF*nx*ny
print('nodes:', nx*ny, ' elements:', num_elements, ' degrees of freedom:', N)

nodes: 800  elements: 750  degrees of freedom: 8000


## Buckling under a constant membrane force

The geometric stiffness matrix of `update_KG_constant_stress` is that of
the uniform axial membrane force $N_{xx} = -1$ N/m, without a static
analysis: the prebuckling state is taken as a membrane state, as in the
result BACCUS-C4 of [Geier2002], and the boundary conditions enter only
through $\boldsymbol{K}_{C0}$. The ends are clamped, $u = v = w = w_{,x} =
0$, the conditions CC1 of [Castro2014], Table 1. They are imposed at the
nodes of the ends together with the derivatives along the edge, $u_{,y}$,
$v_{,y}$, $w_{,y}$ and $w_{,xy}$, so that they hold along the whole edge
and not only at the nodes, see [Assembling a model](../assembling.rst). The linear
buckling problem, see [Analyses](../analyses.rst),

$$
\left( \boldsymbol{K}_{C0} + \lambda \boldsymbol{K}_G \right)
\boldsymbol{\phi} = \boldsymbol{0}
$$

is solved for the lowest load multipliers with the shift-invert mode
`'buckling'` of `eigsh`, four of them, from a fixed starting vector and
with the shift at half the load multiplier of an estimate of the buckling
load, here the 192 kN of [Geier2002]: the transformed eigenvalues near the
lowest load multipliers are then well separated. The lowest load
multiplier $\lambda_1$ is checked by the residual of its eigenpair and by
the number of negative pivots of the $\boldsymbol{L} \boldsymbol{D}
\boldsymbol{L}^\top$ factorization of $\boldsymbol{K}_{C0} + s
\boldsymbol{K}_G$ for $s = (1 - 10^{-6}) \lambda_1$, which by Sylvester's
law of inertia is the number of load multipliers between 0 and $s$: zero
when no lower one is missing.
The critical load is $P_{cr} = \lambda_1 N_{xx} 2 \pi R$. The function below runs the whole
analysis with the namespace and the element class of one kinematics.

In [4]:
nint = 4
points, weights = get_points_weights(nint=nint)
Nxx = -1.  # N/m
P_est = 192e3  # N, estimate of the buckling load for the shift

def negative_pivots(A):
    """number of negative eigenvalues of the symmetric matrix A, from the
    LDL^T factorization with a symmetric permutation"""
    lu = splu(A.tocsc(), permc_spec='MMD_AT_PLUS_A', diag_pivot_thresh=0.,
              options=dict(SymmetricMode=True))
    if not np.array_equal(lu.perm_r, lu.perm_c):
        raise RuntimeError('row interchange, the count is not the inertia')
    return int(np.count_nonzero(lu.U.diagonal() < 0))

def lowest_load_multiplier(Kuu, KGuu, estimate, k=4):
    """lowest load multiplier and its mode, with the residual of the pair and
    the number of load multipliers below it"""
    eigvals, eigvecs = eigsh(A=Kuu, k=k, which='LM', M=KGuu,
                             sigma=-0.5*estimate, mode='buckling',
                             v0=np.ones(Kuu.shape[0]))
    load_mult = -eigvals
    i = np.argmin(load_mult)
    lm, phi = load_mult[i], eigvecs[:, i]
    residual = norm(Kuu @ phi + lm*(KGuu @ phi))/norm(Kuu @ phi)
    below = negative_pivots(Kuu + (1 - 1e-6)*lm*KGuu)
    return lm, phi, residual, below

def buckling_constant_stress(module, Element):
    elements = []
    for k, (n1, n2, n3, n4) in enumerate(zip(n1s, n2s, n3s, n4s)):
        shell = Element(nint)
        shell.n1, shell.n2, shell.n3, shell.n4 = n1, n2, n3, n4
        shell.c1 = DOF*nid_pos[n1]
        shell.c2 = DOF*nid_pos[n2]
        shell.c3 = DOF*nid_pos[n3]
        shell.c4 = DOF*nid_pos[n4]
        shell.R = R
        shell.lex = L/(nx - 1)
        shell.ley = b/ny
        assign_constant_ABD(shell, prop)
        shell.init_k_KC0 = k*module.KC0_SPARSE_SIZE
        shell.init_k_KG = k*module.KG_SPARSE_SIZE
        elements.append(shell)

    Kr = np.zeros(module.KC0_SPARSE_SIZE*num_elements, dtype=module.INT)
    Kc = np.zeros_like(Kr)
    Kv = np.zeros(module.KC0_SPARSE_SIZE*num_elements, dtype=module.DOUBLE)
    KGr = np.zeros(module.KG_SPARSE_SIZE*num_elements, dtype=module.INT)
    KGc = np.zeros_like(KGr)
    KGv = np.zeros(module.KG_SPARSE_SIZE*num_elements, dtype=module.DOUBLE)
    for shell in elements:
        module.update_KC0(shell, points, weights, Kr, Kc, Kv)
        module.update_KG_constant_stress(shell, points, weights, KGr, KGc, KGv,
                                         Nxx, 0., 0.)
    KC0 = coo_matrix((Kv, (Kr, Kc)), shape=(N, N)).tocsc()
    KG = coo_matrix((KGv, (KGr, KGc)), shape=(N, N)).tocsc()

    # clamped ends: u = v = w = w_x = 0, with u_y, v_y, w_y, w_xy = 0
    bk = np.zeros(N, dtype=bool)
    ends = isclose(x, 0) | isclose(x, L)
    for dof in (0, 2, 3, 5, 6, 7, 8, 9):
        bk[dof::DOF] = ends
    bu = ~bk

    lm, phi, residual, below = lowest_load_multiplier(
        KC0[bu][:, bu], KG[bu][:, bu], estimate=P_est/(abs(Nxx)*b))
    mode = np.zeros(N)
    mode[bu] = phi
    return lm*Nxx*b, mode, residual, below

results = {}
for name, (module, Element) in KINEMATICS.items():
    Pcr, mode, residual, below = buckling_constant_stress(module, Element)
    results[name] = Pcr, mode
    print(f'{name:8s} P_cr = {Pcr/1e3:.2f} kN, residual {residual:.1e}, '
          f'load multipliers below: {below}')

Donnell  P_cr = -201.10 kN, residual 4.0e-10, load multipliers below: 0


Sanders  P_cr = -199.87 kN, residual 4.2e-09, load multipliers below: 0


## Buckling modes

$w$ of the first mode on the developed mid-surface, for each kinematics.

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(7, 5.5), sharex=True)
for ax, (name, (Pcr, mode)) in zip(axes, results.items()):
    w = mode[6::DOF].reshape(nx, ny)
    c = ax.contourf(ymesh*1e3, xmesh*1e3, w/np.abs(w).max(), levels=50,
                    cmap='coolwarm')
    ax.set_aspect('equal')
    ax.set_ylabel('$x$ (mm)')
    ax.set_title(f'{name}-type kinematics, $P_{{cr}}$ = {abs(Pcr)/1e3:.1f} kN')
axes[-1].set_xlabel('$y$ (mm), around the circumference')
fig.colorbar(c, ax=axes, label='$w / |w|_{max}$')
plt.show()

## Comparison

The reference is the buckling load of Z33 of Geier et al. [Geier2002],
Table 1, 192 kN, the result BACCUS-C4: built-in edges, deep shell theory
and a homogeneous membrane prebuckling state; the tests of the same paper
gave 173 kN and 184 kN. The table also gives the values that the tests of
the two elements store (regression values of the same model).

In [6]:
P_ref = 192e3  # N, Geier et al. (2002), Table 1, BACCUS-C4
stored = {'Donnell': 201104., 'Sanders': 199872.11}  # N, the tests
rows = []
for name, (Pcr, mode) in results.items():
    rows.append({'kinematics': name, 'P_cr (kN)': abs(Pcr)/1e3,
                 'reference (kN)': P_ref/1e3,
                 'source': 'Geier et al. (2002), Table 1',
                 'rel. diff. (%)': 100*(abs(Pcr) - P_ref)/P_ref})
    rows.append({'kinematics': name, 'P_cr (kN)': abs(Pcr)/1e3,
                 'reference (kN)': stored[name]/1e3,
                 'source': 'regression value of the test',
                 'rel. diff. (%)': 100*(abs(Pcr) - stored[name])/stored[name]})
table(rows)

| kinematics | P_cr (kN) | reference (kN) | source | rel. diff. (%) |
|---|---|---|---|---|
| Donnell | 201.1 | 192 | Geier et al. (2002), Table 1 | 4.7419 |
| Donnell | 201.1 | 201.1 | regression value of the test | 0.00018657 |
| Sanders | 199.87 | 192 | Geier et al. (2002), Table 1 | 4.1001 |
| Sanders | 199.87 | 199.87 | regression value of the test | 3.9396e-07 |

In [7]:
d = abs(results['Sanders'][0])/abs(results['Donnell'][0]) - 1
print(f'Sanders-type relative to Donnell-type: {100*d:.2f} %')

Sanders-type relative to Donnell-type: -0.61 %


The two elements are 4.1 and 4.7 % above the reference of [Geier2002], a
difference that contains the discretisation error of 50 nodes around the
circumference and the different radius of reference: here the mid-surface,
in [Geier2002] the inner surface. The Sanders-type element gives a load
smaller by less than 1 %, as expected for a mode with many circumferential
waves.

## References

- **[Castro2014]** Castro, S. G. P., Mittelstedt, C., Monteiro, F. A. C., Arbelo, M. A.,
  Ziegmann, G., and Degenhardt, R., Linear buckling predictions of unstiffened
  laminated composite cylinders and cones under various loading and boundary
  conditions using semi-analytical models, Composite Structures, 118, 303-315,
  2014, doi:10.1016/j.compstruct.2014.07.037.
- **[Geier2002]** Geier, B., Meyer-Piening, H.-R., and Zimmermann, R., On the influence of
  laminate stacking on buckling of composite cylindrical shells subjected to
  axial compression, Composite Structures, 55(4), 467-474, 2002,
  doi:10.1016/S0263-8223(01)00175-1.
- **[GeierSingh1997]** Geier, B., and Singh, G., Some simple solutions for buckling loads of thin
  and moderately thick cylindrical shells and panels made of laminated
  composite material, Aerospace Science and Technology, 1(1), 47-63, 1997,
  doi:10.1016/S1270-9638(97)90023-7.
- **[Donnell1933]** Donnell, L. H., Stability of thin-walled tubes under torsion, NACA
  Report 479, 1933.
- **[Sanders1959]** Sanders, J. L., An improved first-approximation theory for thin shells,
  NASA Technical Report R-24, 1959.
- **[Koiter1960]** Koiter, W. T., A consistent first approximation in the general theory of
  thin elastic shells, Proceedings of the IUTAM Symposium on the Theory of
  Thin Elastic Shells, Delft, 1959, North-Holland, Amsterdam, 1960,
  pp. 12-33.